In [ ]:
# Cell 1 — Verify GPU
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM")

True
NVIDIA A100-SXM4-80GB
85.1 GB VRAM


In [ ]:
# Fix numpy FIRST before anything else
!pip install -q "numpy>=2.0"

# Then install libraries
!pip install -q transformers==4.44.0
!pip install -q datasets==2.20.0
!pip install -q peft==0.12.0
!pip install -q trl==0.9.6
!pip install -q bitsandbytes==0.45.3
!pip install -q accelerate==0.33.0
!pip install -q huggingface_hub

print("✅ Libraries installed!")
print("⚠️  NOW: Runtime → Restart Session")
print("⚠️  THEN: Skip this cell, run Cell 3 onwards")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 115.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 31.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.5.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 21.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.8/245.8 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━

In [ ]:
# Verify all our libraries imported correctly
import transformers
import datasets
import peft
import trl
import bitsandbytes
import accelerate
import huggingface_hub

print(f"transformers:    {transformers.__version__}")
print(f"datasets:        {datasets.__version__}")
print(f"peft:            {peft.__version__}")
print(f"trl:             {trl.__version__}")
print(f"bitsandbytes:    {bitsandbytes.__version__}")
print(f"accelerate:      {accelerate.__version__}")
print(f"huggingface_hub: {huggingface_hub.__version__}")

print("\n✅ All imports successful — ready to train!")

transformers:    4.44.0
datasets:        2.20.0
peft:            0.12.0
trl:             0.9.6
bitsandbytes:    0.45.3
accelerate:      0.33.0
huggingface_hub: 0.36.2

✅ All imports successful — ready to train!


In [33]:
from huggingface_hub import login

HF_USERNAME = "ninadp"
MODEL_REPO = f"ninadp/marathi-mitra-phi3"

login(token=HF_TOKEN)
print(f"✅ Logged in!")
print(f"✅ Model will be saved to: {MODEL_REPO}")

✅ Logged in!
✅ Model will be saved to: ninadp/marathi-mitra-phi3


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL A — UTILITIES (run once after model loads)
# No external API dependencies — fully self-contained
# ═══════════════════════════════════════════════════════════
import os
import re
import json
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# ── Constants ─────────────────────────────────────────────
TEST_WORDS = ["butterfly", "mother", "rain", "elephant", "school"]

GOLDEN = {
    "butterfly": {
        "marathi":       "फुलपाखरू",
        "pronunciation": "Phul-pakh-roo",
    },
    "mother": {
        "marathi":       "आई",
        "pronunciation": "Aa-ee",
    },
    "rain": {
        "marathi":       "पाऊस",
        "pronunciation": "Paa-oos",
    },
    "elephant": {
        "marathi":       "हत्ती",
        "pronunciation": "Hat-tee",
    },
    "school": {
        "marathi":       "शाळा",
        "pronunciation": "Shaa-la",
    },
}

ALL_RESULTS = {}


# ── Generate one response ─────────────────────────────────
def generate(word, model, tokenizer):
    """Run one word through fine-tuned model and return output."""
    prompt = f"""### Instruction:
You are Marathi Mitra, a friendly Marathi teacher for kids. \
When given an English word, teach it in Marathi with the word \
in Devanagari script, pronunciation, a simple story sentence, \
and a fun fact. Always be encouraging and kid-friendly.

### Input:
Teach me the Marathi word for: {word}

### Response:
"""
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.3,
        )

    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return full_text.split("### Response:")[-1].strip()


# ── Extract fields from model output ─────────────────────
def extract_fields(output: str) -> dict:
    """Parse model output and extract Marathi word and pronunciation."""
    fields = {}

    marathi_match = re.search(r"is \*\*([^*]+)\*\*", output)
    fields["marathi_word"] = (
        marathi_match.group(1).strip() if marathi_match else ""
    )

    pronun_match = re.search(r"How to say it:\*?\*?\s*(.+)", output)
    fields["pronunciation"] = (
        pronun_match.group(1).strip() if pronun_match else ""
    )

    return fields


# ── Evaluate one output (2 criteria) ─────────────────────
def evaluate_output(output: str, word: str) -> tuple:
    """
    Two criteria evaluation:

    Criteria 1 — Field presence  (40%)
    → Are all 5 expected sections present?

    Criteria 2 — Exact match     (60%)
    → Is the Marathi word correct?
    → Is the pronunciation correct?
    """
    golden = GOLDEN.get(word, {})
    fields = extract_fields(output)

    # ── Criteria 1: Field presence ─────────────────────────
    presence = {
        "Has Devanagari word":     bool(
            re.search(r"[\u0900-\u097F]", output)
        ),
        "Has pronunciation guide": "How to say" in output,
        "Has example sentence":    "Example sentence" in output,
        "Has fun fact":            "Fun Fact" in output,
        "Has correct emojis":      "🌟" in output and "📢" in output,
    }
    presence_score = sum(presence.values()) / len(presence) * 100

    # ── Criteria 2: Exact match ────────────────────────────
    exact = {}
    if golden:
        exact["Marathi word correct"] = (
            golden["marathi"] in fields["marathi_word"] or
            golden["marathi"] in output
        )
        model_p  = fields["pronunciation"].lower().replace(" ", "").replace("-", "")
        golden_p = golden["pronunciation"].lower().replace(" ", "").replace("-", "")
        exact["Pronunciation correct"] = model_p == golden_p

    exact_score = (
        sum(exact.values()) / len(exact) * 100
        if exact else 0
    )

    # ── Combined weighted score ────────────────────────────
    combined = (
        presence_score * 0.40 +
        exact_score    * 0.60
    )

    criteria = {
        "presence": presence,
        "exact":    exact,
    }

    return criteria, combined, fields


# ── Evaluate all 5 test words ─────────────────────────────
def evaluate_model(model, tokenizer, exp_name):
    """
    Test model on all 5 words.
    Shows actual output + both criteria per word.
    Shows combined score at end.
    Stores in ALL_RESULTS.
    """
    print(f"\n{'═'*60}")
    print(f"EVALUATION: {exp_name}")
    print(f"{'═'*60}")

    outputs = {}
    scores  = {}
    total   = 0

    for word in TEST_WORDS:
        print(f"\n{'─'*60}")
        print(f"📝 Word: {word.upper()}")
        print(f"{'─'*60}")

        try:
            output = generate(word, model, tokenizer)
        except Exception as e:
            output = f"[ERROR: {e}]"

        outputs[word] = output
        print(f"\n🤖 Output:\n{output}")

        criteria, score, fields = evaluate_output(output, word)
        scores[word] = score
        total += score

        # ── Criteria 1 ─────────────────────────────────────
        print(f"\n📊 Criteria 1 — Field Presence (40%):")
        for criterion, passed in criteria["presence"].items():
            print(f"   {'✅' if passed else '❌'} {criterion}")

        # ── Criteria 2 ─────────────────────────────────────
        print(f"\n📊 Criteria 2 — Exact Match (60%):")
        golden = GOLDEN.get(word, {})
        print(f"   Expected Marathi:       {golden.get('marathi', '?')}")
        print(f"   Model Marathi:          {fields['marathi_word'] or 'not found'}")
        print(f"   Expected Pronunciation: {golden.get('pronunciation', '?')}")
        print(f"   Model Pronunciation:    {fields['pronunciation'] or 'not found'}")
        for criterion, passed in criteria["exact"].items():
            print(f"   {'✅' if passed else '❌'} {criterion}")

        print(f"\n   ─────────────────────────────────────────")
        print(f"   Word score: {score:.1f}%")
        print(f"   (presence 40% + exact match 60%)")

    # ── Summary ────────────────────────────────────────────
    combined = total / len(TEST_WORDS)

    print(f"\n{'═'*60}")
    print(f"SCORES SUMMARY")
    print(f"{'═'*60}")
    for word, s in scores.items():
        bar = "█" * int(s/10) + "░" * (10-int(s/10))
        print(f"  {word:<12} {bar} {s:.0f}%")
    print(f"{'─'*60}")
    combined_bar = "█" * int(combined/10) + "░" * (10-int(combined/10))
    print(f"  {'COMBINED':<12} {combined_bar} {combined:.1f}%")
    print(f"{'═'*60}")

    ALL_RESULTS[exp_name] = {
        "outputs":  outputs,
        "scores":   scores,
        "combined": combined,
    }
    return outputs, combined


# ── Load fresh base model ─────────────────────────────────
def load_fresh_model():
    """Load fresh quantized Phi-3 ready for LoRA."""
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    m = AutoModelForCausalLM.from_pretrained(
        "microsoft/Phi-3-mini-4k-instruct",
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    return prepare_model_for_kbit_training(m)


# ── Apply LoRA adapters ───────────────────────────────────
def apply_lora(model, r, lora_alpha):
    """Apply LoRA adapters with given rank and scaling."""
    config = LoraConfig(
        r=r,
        lora_alpha=lora_alpha,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    m = get_peft_model(model, config)
    m.print_trainable_parameters()
    return m


# ── Train model ───────────────────────────────────────────
def train_model(model, exp_name, epochs, learning_rate):
    """Train model with given config. Prints loss progression."""
    args = SFTConfig(
        output_dir=f"./results/{exp_name}",
        num_train_epochs=epochs,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=learning_rate,
        fp16=True,
        logging_steps=10,
        max_seq_length=512,
        dataset_text_field="text",
        warmup_ratio=0.1,
        save_strategy="no",
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=hf_dataset,
        tokenizer=tokenizer,
    )

    print(f"🚀 Training {exp_name}...")
    print(f"   lr={learning_rate}, epochs={epochs}")
    trainer.train()

    print("\nLoss progression:")
    for log in trainer.state.log_history:
        if "loss" in log:
            print(f"  Step {log['step']:3d} → Loss: {log['loss']:.4f}")

    step_losses = [
        log["loss"] for log in trainer.state.log_history
        if "loss" in log
    ]
    final = step_losses[-1]
    print(f"\n✅ Done! Final loss: {final:.4f}")

    if exp_name not in ALL_RESULTS:
        ALL_RESULTS[exp_name] = {}
    ALL_RESULTS[exp_name]["final_loss"]    = round(final, 4)
    ALL_RESULTS[exp_name]["learning_rate"] = learning_rate
    ALL_RESULTS[exp_name]["epochs"]        = epochs

    return trainer


# ── Save results to disk ──────────────────────────────────
def save_results(path="./results/all_results.json"):
    """Save ALL_RESULTS to disk — survives session restarts."""
    os.makedirs("./results", exist_ok=True)
    serializable = {
        name: {
            "combined":      data.get("combined", 0),
            "final_loss":    data.get("final_loss", None),
            "learning_rate": data.get("learning_rate", None),
            "epochs":        data.get("epochs", None),
        }
        for name, data in ALL_RESULTS.items()
    }
    with open(path, "w") as f:
        json.dump(serializable, f, indent=2)
    print(f"✅ Results saved → {path}")


# ── Load results from disk ────────────────────────────────
def load_results(path="./results/all_results.json"):
    """Load ALL_RESULTS from disk after session restart."""
    if not os.path.exists(path):
        print("No saved results found — starting fresh")
        return
    with open(path) as f:
        saved = json.load(f)
    for name, data in saved.items():
        if name not in ALL_RESULTS:
            ALL_RESULTS[name] = data
    print(f"✅ Restored {len(saved)} experiment results")
    print_comparison_table()


# ── Compare all experiments ───────────────────────────────
def print_comparison_table():
    """Print full comparison table of all experiments."""
    if not ALL_RESULTS:
        print("No experiments run yet.")
        return

    print(f"\n{'═'*75}")
    print("EXPERIMENT COMPARISON")
    print(f"{'═'*75}")
    print(
        f"{'Experiment':<35} {'LR':<8} {'Epochs':<8} "
        f"{'Loss':<8} {'Score':<8} Bar"
    )
    print(f"{'─'*75}")

    for name, result in ALL_RESULTS.items():
        score  = result.get("combined", 0)
        loss   = result.get("final_loss", None)
        lr     = result.get("learning_rate", None)
        epochs = result.get("epochs", None)
        bar    = "█" * int(score/10) + "░" * (10-int(score/10))

        loss_str   = f"{loss:.4f}" if isinstance(loss, float) else "N/A"
        lr_str     = f"{lr:.0e}"   if isinstance(lr, float)   else "N/A"
        epochs_str = str(epochs)   if isinstance(epochs, int) else "N/A"

        print(
            f"{name:<35} {lr_str:<8} {epochs_str:<8} "
            f"{loss_str:<8} {score:<8.1f} {bar}"
        )

    print(f"{'═'*75}")
    best       = max(ALL_RESULTS, key=lambda x: ALL_RESULTS[x].get("combined", 0))
    best_score = ALL_RESULTS[best]["combined"]
    best_loss  = ALL_RESULTS[best].get("final_loss", "N/A")
    print(f"🏆 Best: {best}")
    print(f"   Score: {best_score:.1f}%")
    if isinstance(best_loss, float):
        print(f"   Loss:  {best_loss:.4f}")


# ── Verify setup ──────────────────────────────────────────
print("✅ CELL A loaded — all functions ready!")
print(f"   Test words: {TEST_WORDS}")
print()
print("   Functions available:")
print("   → generate()               model inference")
print("   → extract_fields()         parse output")
print("   → evaluate_output()        2 criteria scoring")
print("   → evaluate_model()         test all 5 words")
print("   → load_fresh_model()       fresh Phi-3")
print("   → apply_lora()             LoRA adapters")
print("   → train_model()            SFTTrainer wrapper")
print("   → save_results()           persist to disk")
print("   → load_results()           restore from disk")
print("   → print_comparison_table() results summary")
print()
print("   Evaluation criteria:")
print("   → Criteria 1: Field presence  40%")
print("   → Criteria 2: Exact match     60%")

✅ CELL A loaded — all functions ready!
   Test words: ['butterfly', 'mother', 'rain', 'elephant', 'school']

   Functions available:
   → generate()               model inference
   → extract_fields()         parse output
   → evaluate_output()        2 criteria scoring
   → evaluate_model()         test all 5 words
   → load_fresh_model()       fresh Phi-3
   → apply_lora()             LoRA adapters
   → train_model()            SFTTrainer wrapper
   → save_results()           persist to disk
   → load_results()           restore from disk
   → print_comparison_table() results summary

   Evaluation criteria:
   → Criteria 1: Field presence  40%
   → Criteria 2: Exact match     60%


In [ ]:
# ═══════════════════════════════════════════════════════════
# Load Dataset
# ═══════════════════════════════════════════════════════════
import json
import urllib.request
from datasets import Dataset    # ← Dataset class, not data variable

GITHUB_USERNAME = "ninadparab"
url = f"https://raw.githubusercontent.com/ninadparab/marathi-mitra/main/data/vocabulary_dataset.json"

# Step 1 — Download raw JSON
with urllib.request.urlopen(url) as response:
    raw_data = json.loads(response.read().decode("utf-8"))

print(f"✅ Downloaded {len(raw_data)} examples from GitHub")
print(f"✅ Type of raw_data: {type(raw_data)}")   # should be <class 'list'>

# Step 2 — Convert list → HuggingFace Dataset
hf_dataset = Dataset.from_list(raw_data)

print(f"✅ hf_dataset ready: {len(hf_dataset)} examples")
print(f"✅ Type: {type(hf_dataset)}")             # should be datasets.arrow_dataset.Dataset
print(f"✅ Columns: {hf_dataset.column_names}")

✅ Downloaded 30 examples from GitHub
✅ Type of raw_data: <class 'list'>
✅ hf_dataset ready: 30 examples
✅ Type: <class 'datasets.arrow_dataset.Dataset'>
✅ Columns: ['word', 'marathi', 'instruction', 'input', 'output', 'text']


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

# 4-bit quantization — makes model fit in T4 VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading Phi-3 Mini... (takes 3-5 mins)")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"✅ Model loaded!")
print(f"✅ Size in memory: {model.get_memory_footprint() / 1e9:.2f} GB")

Loading Phi-3 Mini... (takes 3-5 mins)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

✅ Model loaded!
✅ Size in memory: 2.21 GB


In [ ]:
# ═══════════════════════════════════════════════════════════
# BASELINE (before fine-tuning)
# ═══════════════════════════════════════════════════════════

baseline_outputs, baseline_score = evaluate_model(
    model, tokenizer, "exp0_baseline"
)


════════════════════════════════════════════════════════════
EVALUATION: exp0_baseline
════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────
📝 Word: BUTTERFLY
────────────────────────────────────────────────────────────

🤖 Output:
सुंदर कायची! (Sundar Kayachi!) This is how you say "buttery" or related to something beautiful like our friend here - 'फलविजन' which means Butterflies ('पत्थरविजन') written as **प**(pa) + *त*(-ta), where each letter represents its sound when spoken out loud; /p/at̪ɾəʋɪd͡ʒn/. Here’a tale about this lovely insect goes thus – Once upon time there was tiny caterpie named Bala who loved flying around colorful flowers more than anything else because she felt free just being up high among

📊 Criteria 1 — Field Presence (40%):
   ✅ Has Devanagari word
   ❌ Has pronunciation guide
   ❌ Has example sentence
   ❌ Has fun fact
   ❌ Has correct emojis

📊 Criteria 2 — Exact Match (60%):
   Expected Marat

In [ ]:
# ═══════════════════════════════════════════════════════════
# EXPERIMENT 1 TRAINING
# lr=2e-4, epochs=5, r=16
# ═══════════════════════════════════════════════════════════
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

def load_fresh_model():
    """Load a fresh quantized Phi-3 ready for LoRA."""
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    m = AutoModelForCausalLM.from_pretrained(
        "microsoft/Phi-3-mini-4k-instruct",
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    return prepare_model_for_kbit_training(m)


def apply_lora(model, r, lora_alpha):
    """Apply LoRA adapters to model."""
    config = LoraConfig(
        r=r,
        lora_alpha=lora_alpha,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    m = get_peft_model(model, config)
    m.print_trainable_parameters()
    return m


def train_model(model, exp_name, epochs, learning_rate):
    """Train model and print loss progression."""
    args = SFTConfig(
        output_dir=f"./results/{exp_name}",
        num_train_epochs=epochs,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=learning_rate,
        fp16=True,
        logging_steps=10,
        max_seq_length=512,
        dataset_text_field="text",
        warmup_ratio=0.1,
        report_to="none",
    )
    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=hf_dataset,
        tokenizer=tokenizer,
    )
    print(f"🚀 Training {exp_name}...")
    trainer.train()

    # Print loss history
    print("\nLoss progression:")
    for log in trainer.state.log_history:
        if "loss" in log:
            print(f"  Step {log['step']:3d} → Loss: {log['loss']:.4f}")

    final_loss = trainer.state.log_history[-1].get("train_loss", None)
    print(f"\n✅ Training complete! Final loss: {final_loss:.4f}")
    return trainer


# ── Run Experiment 1 ─────────────────────────────────────────
print("Loading fresh model for Experiment 1...")
model1 = load_fresh_model()
model1 = apply_lora(model1, r=16, lora_alpha=32)
trainer1 = train_model(model1, "exp1_lr2e4_epochs5_r16", epochs=5, learning_rate=2e-4)

Loading fresh model for Experiment 1...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 3,145,728 || all params: 3,824,225,280 || trainable%: 0.0823


Map:   0%|          | 0/30 [00:00<?, ? examples/s]

🚀 Training exp1_lr2e4_epochs5_r16...


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.588500



Loss progression:
  Step  10 → Loss: 1.5885

✅ Training complete! Final loss: 1.4864


In [ ]:
# ═══════════════════════════════════════════════════════════
# EVALUATE EXPERIMENT 1
# ═══════════════════════════════════════════════════════════

exp1_outputs, exp1_score = evaluate_model(
    model1, tokenizer, "exp1_lr2e4_epochs5_r16"
)
print_comparison_table()


════════════════════════════════════════════════════════════
EVALUATION: exp1_lr2e4_epochs5_r16
════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────
📝 Word: BUTTERFLY
────────────────────────────────────────────────────────────


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)



🤖 Output:
🌼 **Marathi Word** 🦋 *Pheral* (pronounced as pherāl)  
✨ Here's how you can use this beautiful insect name! "फेरालच्यों किती मदत आपण?" which means 'What kind of help does your friend give?' Imagine if Pheral is helping its friends by spreading pollen from flower to flowers like tiny messengers carrying sweet treats all around their garden home - that’ieves us right? Fun Fact Time!! Did You Know?: Buttermilk was once used on farms after harvest season because cows would

📊 Criteria 1 — Field Presence (40%):
   ✅ Has Devanagari word
   ❌ Has pronunciation guide
   ❌ Has example sentence
   ✅ Has fun fact
   ❌ Has correct emojis

📊 Criteria 2 — Exact Match (60%):
   Expected Marathi:       फुलपाखरू
   Model Marathi:          not found
   Expected Pronunciation: Phul-pakh-roo
   Model Pronunciation:    not found
   ❌ Marathi word correct
   ❌ Pronunciation correct

   ─────────────────────────────────────────
   Word score: 16.0%
   (presence 40% + exact match 60%)

────────────

In [ ]:
# ═══════════════════════════════════════════════════════════
# EXPERIMENT 2 TRAINING
# lr=2e-4, epochs=25, r=16  ← more epochs
# ═══════════════════════════════════════════════════════════

print("Loading fresh model for Experiment 2...")
model2 = load_fresh_model()
model2 = apply_lora(model2, r=16, lora_alpha=32)
trainer2 = train_model(model2, "exp2_lr2e4_epochs25_r16", epochs=25, learning_rate=2e-4)

Loading fresh model for Experiment 2...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 3,145,728 || all params: 3,824,225,280 || trainable%: 0.0823


Map:   0%|          | 0/30 [00:00<?, ? examples/s]

🚀 Training exp2_lr2e4_epochs25_r16...


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.664300
20,1.027700
30,0.464000
40,0.337700
50,0.280900
60,0.241000
70,0.204200



Loss progression:
  Step  10 → Loss: 1.6643
  Step  20 → Loss: 1.0277
  Step  30 → Loss: 0.4640
  Step  40 → Loss: 0.3377
  Step  50 → Loss: 0.2809
  Step  60 → Loss: 0.2410
  Step  70 → Loss: 0.2042

✅ Training complete! Final loss: 0.5766


In [ ]:
# ═══════════════════════════════════════════════════════════
# EVALUATE EXPERIMENT 2
# ═══════════════════════════════════════════════════════════

exp2_outputs, exp2_score = evaluate_model(
    model2, tokenizer, "exp2_lr2e4_epochs25_r16"
)
print_comparison_table()


════════════════════════════════════════════════════════════
EVALUATION: exp2_lr2e4_epochs25_r16
════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────
📝 Word: BUTTERFLY
────────────────────────────────────────────────────────────


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)



🤖 Output:
🌟 **BUTTERFLY** in Marathi is **फुल्हा**

📢 How to say it: Phulhaa

💐 Example sentence:
चंगीरे मधकतो नवजात आयड़णा.
*(The new flowers attract beautiful wings.)*

📊 Criteria 1 — Field Presence (40%):
   ✅ Has Devanagari word
   ✅ Has pronunciation guide
   ✅ Has example sentence
   ❌ Has fun fact
   ✅ Has correct emojis

📊 Criteria 2 — Exact Match (60%):
   Expected Marathi:       फुलपाखरू
   Model Marathi:          फुल्हा
   Expected Pronunciation: Phul-pakh-roo
   Model Pronunciation:    Phulhaa
   ❌ Marathi word correct
   ❌ Pronunciation correct

   ─────────────────────────────────────────
   Word score: 32.0%
   (presence 40% + exact match 60%)

────────────────────────────────────────────────────────────
📝 Word: MOTHER
────────────────────────────────────────────────────────────

🤖 Output:
🌟 **MOTHER** in MARATHI is मातृ

📢 How to say it: Maaatrri

✍️ In a sentence:
आंगणी जेवरा सुबन्दोलियर होतात.
*(The Bengali woman' instruction was cut off at "Bengal")* I will complete

In [ ]:
# ═══════════════════════════════════════════════════════════
# EXPERIMENT 3 TRAINING
# lr=1e-4, epochs=25, r=16  ← lower learning rate
# ═══════════════════════════════════════════════════════════

model3   = load_fresh_model()
model3   = apply_lora(model3, r=16, lora_alpha=32)
trainer3 = train_model(
    model3,
    exp_name="exp3_lr1e4_epochs25_r16",
    epochs=25,
    learning_rate=1e-4,
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 3,145,728 || all params: 3,824,225,280 || trainable%: 0.0823


Map:   0%|          | 0/30 [00:00<?, ? examples/s]

🚀 Training exp3_lr1e4_epochs25_r16...


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.710500
20,1.418100
30,0.979000
40,0.585700
50,0.453700
60,0.409600
70,0.373500



Loss progression:
  Step  10 → Loss: 1.7105
  Step  20 → Loss: 1.4181
  Step  30 → Loss: 0.9790
  Step  40 → Loss: 0.5857
  Step  50 → Loss: 0.4537
  Step  60 → Loss: 0.4096
  Step  70 → Loss: 0.3735

✅ Training complete! Final loss: 0.8167


In [ ]:
# ═══════════════════════════════════════════════════════════
# EVALUATE EXPERIMENT 3
# ═══════════════════════════════════════════════════════════

exp3_outputs, exp3_score = evaluate_model(
    model3, tokenizer, "exp3_lr1e4_epochs25_r16"
)
print_comparison_table()


════════════════════════════════════════════════════════════
EVALUATION: exp3_lr1e4_epochs25_r16
════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────
📝 Word: BUTTERFLY
────────────────────────────────────────────────────────────


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)



🤖 Output:
🌟 **BUTTERFLY** 🦋 In Marathi we say: **फलतारी**  
*How to say it: Phaltaaree (Fahl -tah – reh)*   
*(Example Sentence): मोहिणेच्या पुस्तकावज आगे खूप अनंत।*    
Did you know? Butterflies have taste buds on their feet! They use them like our tongue does when they sip nectar from flowers... Isn'ht that amazin’?? Let's learn more words together!!

📊 Criteria 1 — Field Presence (40%):
   ✅ Has Devanagari word
   ✅ Has pronunciation guide
   ❌ Has example sentence
   ❌ Has fun fact
   ❌ Has correct emojis

📊 Criteria 2 — Exact Match (60%):
   Expected Marathi:       फुलपाखरू
   Model Marathi:          not found
   Expected Pronunciation: Phul-pakh-roo
   Model Pronunciation:    Phaltaaree (Fahl -tah – reh)*
   ❌ Marathi word correct
   ❌ Pronunciation correct

   ─────────────────────────────────────────
   Word score: 16.0%
   (presence 40% + exact match 60%)

────────────────────────────────────────────────────────────
📝 Word: MOTHER
─────────────────────────────────────────────

In [ ]:
# ═══════════════════════════════════════════════════════════
# EXPERIMENT 4 TRAINING
# lr=2e-4, epochs=25, r=32  ← bigger LoRA rank
# ═══════════════════════════════════════════════════════════

model4   = load_fresh_model()
model4   = apply_lora(model4, r=32, lora_alpha=64)
trainer4 = train_model(
    model4,
    exp_name="exp4_lr2e4_epochs25_r32",
    epochs=25,
    learning_rate=2e-4,
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 6,291,456 || all params: 3,827,371,008 || trainable%: 0.1644


Map:   0%|          | 0/30 [00:00<?, ? examples/s]

🚀 Training exp4_lr2e4_epochs25_r32...


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.599700
20,0.734200
30,0.352400
40,0.258100
50,0.191600
60,0.145200
70,0.107900



Loss progression:
  Step  10 → Loss: 1.5997
  Step  20 → Loss: 0.7342
  Step  30 → Loss: 0.3524
  Step  40 → Loss: 0.2581
  Step  50 → Loss: 0.1916
  Step  60 → Loss: 0.1452
  Step  70 → Loss: 0.1079

✅ Training complete! Final loss: 0.4586


In [ ]:
# ═══════════════════════════════════════════════════════════
# EVALUATE EXPERIMENT 4
# ═══════════════════════════════════════════════════════════

exp4_outputs, exp4_score = evaluate_model(
    model4, tokenizer, "exp4_lr2e4_epochs25_r32"
)
print_comparison_table()


════════════════════════════════════════════════════════════
EVALUATION: exp4_lr2e4_epochs25_r32
════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────
📝 Word: BUTTERFLY
────────────────────────────────────────────────────────────


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)



🤖 Output:
🌟 **BUTTERFLY** in Marathi is **पताली**

📢 **How to say it:** Pattaaali

✏️ Example sentence:
फूड्से जगभर मधुकवो अनंच! — *Butterflies taste honey.*

🎉 Fun Fact: हिटलो यशाखाachen padāla आणि बदतलाषणे थे! → **Fascinating!! They can fly very fast!!!

📊 Criteria 1 — Field Presence (40%):
   ✅ Has Devanagari word
   ✅ Has pronunciation guide
   ✅ Has example sentence
   ✅ Has fun fact
   ✅ Has correct emojis

📊 Criteria 2 — Exact Match (60%):
   Expected Marathi:       फुलपाखरू
   Model Marathi:          पताली
   Expected Pronunciation: Phul-pakh-roo
   Model Pronunciation:    Pattaaali
   ❌ Marathi word correct
   ❌ Pronunciation correct

   ─────────────────────────────────────────
   Word score: 40.0%
   (presence 40% + exact match 60%)

────────────────────────────────────────────────────────────
📝 Word: MOTHER
────────────────────────────────────────────────────────────

🤖 Output:
🌟 **MOTHER** in MARATHI is **आपबाती**

📢 **How to say it:** Aap-bata-tee)

✏️ **Example sentence

In [ ]:
# ═══════════════════════════════════════════════════════════
# FINAL SUMMARY TABLE
# Run after all 4 experiments complete
# ═══════════════════════════════════════════════════════════

print_comparison_table()

# Also show best model outputs vs baseline for README
best_exp = max(ALL_RESULTS, key=lambda x: ALL_RESULTS[x].get("combined", 0))

print(f"\n{'═'*60}")
print("BEFORE vs AFTER — BEST MODEL")
print(f"{'═'*60}")

for word in TEST_WORDS:
    before = ALL_RESULTS["exp0_baseline"]["outputs"].get(word, "N/A")
    after  = ALL_RESULTS[best_exp]["outputs"].get(word, "N/A")
    print(f"\n🔤 {word.upper()}")
    print(f"❌ Before:\n{before}")
    print(f"✅ After:\n{after}")
    print("─" * 40)


═══════════════════════════════════════════════════════════════════════════
EXPERIMENT COMPARISON
═══════════════════════════════════════════════════════════════════════════
Experiment                          LR       Epochs   Loss     Score    Bar
───────────────────────────────────────────────────────────────────────────
exp0_baseline                       N/A      N/A      N/A      11.2     █░░░░░░░░░
exp1_lr2e4_epochs5_r16              N/A      N/A      N/A      12.8     █░░░░░░░░░
exp2_lr2e4_epochs25_r16             N/A      N/A      N/A      28.8     ██░░░░░░░░
exp3_lr1e4_epochs25_r16             N/A      N/A      N/A      16.0     █░░░░░░░░░
exp4_lr2e4_epochs25_r32             N/A      N/A      N/A      36.4     ███░░░░░░░
═══════════════════════════════════════════════════════════════════════════
🏆 Best: exp4_lr2e4_epochs25_r32
   Score: 36.4%

════════════════════════════════════════════════════════════
BEFORE vs AFTER — BEST MODEL
═══════════════════════════════════════════

In [ ]:
# Run this after CELL A to restore metadata
ALL_RESULTS["exp1_lr2e4_epochs5_r16"]["final_loss"]    = 1.2896
ALL_RESULTS["exp1_lr2e4_epochs5_r16"]["learning_rate"] = 2e-4
ALL_RESULTS["exp1_lr2e4_epochs5_r16"]["epochs"]        = 5

ALL_RESULTS["exp2_lr2e4_epochs25_r16"]["final_loss"]    = 0.2042
ALL_RESULTS["exp2_lr2e4_epochs25_r16"]["learning_rate"] = 2e-4
ALL_RESULTS["exp2_lr2e4_epochs25_r16"]["epochs"]        = 25

ALL_RESULTS["exp3_lr1e4_epochs25_r16"]["final_loss"]    = 0.3732
ALL_RESULTS["exp3_lr1e4_epochs25_r16"]["learning_rate"] = 1e-4
ALL_RESULTS["exp3_lr1e4_epochs25_r16"]["epochs"]        = 25

ALL_RESULTS["exp4_lr2e4_epochs25_r32"]["final_loss"]    = 0.1079
ALL_RESULTS["exp4_lr2e4_epochs25_r32"]["learning_rate"] = 2e-4
ALL_RESULTS["exp4_lr2e4_epochs25_r32"]["epochs"]        = 25

save_results()
print_comparison_table()

✅ Results saved → ./results/all_results.json

═══════════════════════════════════════════════════════════════════════════
EXPERIMENT COMPARISON
═══════════════════════════════════════════════════════════════════════════
Experiment                          LR       Epochs   Loss     Score    Bar
───────────────────────────────────────────────────────────────────────────
exp0_baseline                       N/A      N/A      N/A      11.2     █░░░░░░░░░
exp1_lr2e4_epochs5_r16              2e-04    5        1.2896   12.8     █░░░░░░░░░
exp2_lr2e4_epochs25_r16             2e-04    25       0.2042   28.8     ██░░░░░░░░
exp3_lr1e4_epochs25_r16             1e-04    25       0.3732   16.0     █░░░░░░░░░
exp4_lr2e4_epochs25_r32             2e-04    25       0.1079   36.4     ███░░░░░░░
═══════════════════════════════════════════════════════════════════════════
🏆 Best: exp4_lr2e4_epochs25_r32
   Score: 36.4%
   Loss:  0.1079


In [ ]:
# Quick check — run this first
print("Models in memory:")
for name, m in [
    ("model  (base)", model),
    ("model1 (exp1)", model1),
    ("model2 (exp2)", model2),
    ("model3 (exp3)", model3),
    ("model4 (exp4)", model4),
]:
    try:
        print(f"  ✅ {name}")
    except NameError:
        print(f"  ❌ {name} — not in memory, retrain first")

Models in memory:
  ✅ model  (base)
  ✅ model1 (exp1)
  ✅ model2 (exp2)
  ✅ model3 (exp3)
  ✅ model4 (exp4)


In [ ]:
# ═══════════════════════════════════════════════════════════
# SAVE BEST MODEL TO HUGGING FACE
# Run after all experiments complete
# ═══════════════════════════════════════════════════════════

# ═══════════════════════════════════════════════════════════
# CELL G — SAVE BEST MODEL TO HUGGING FACE
# ═══════════════════════════════════════════════════════════
import os

# ── Find best experiment ──────────────────────────────────
best_exp   = max(ALL_RESULTS, key=lambda x: ALL_RESULTS[x]["combined"])
best_score = ALL_RESULTS[best_exp]["combined"]

print(f"Best experiment: {best_exp}")
print(f"Best score:      {best_score:.1f}%")
print(f"Saving to:       {MODEL_REPO}")

# ── Map experiment name → model object ────────────────────
exp_model_map = {
    "exp0_baseline":             model,
    "exp1_lr2e4_epochs5_r16":   model1,
    "exp2_lr2e4_epochs25_r16":  model2,
    "exp3_lr1e4_epochs25_r16":  model3,
    "exp4_lr2e4_epochs25_r32":  model4,
}

# Check best model exists in memory
if best_exp not in exp_model_map:
    print(f"❌ {best_exp} not in model map — add it!")
else:
    best_model = exp_model_map[best_exp]

    # ── Step 1: Save locally first ────────────────────────
    # This creates adapter_config.json + adapter weights
    local_path = f"./results/{best_exp}_final"
    os.makedirs(local_path, exist_ok=True)

    print(f"\nStep 1: Saving adapter locally to {local_path}...")
    best_model.save_pretrained(local_path)
    tokenizer.save_pretrained(local_path)

    # Verify files exist
    files = os.listdir(local_path)
    print(f"Files saved: {files}")
    assert "adapter_config.json" in files, "❌ adapter_config.json missing!"
    print("✅ adapter_config.json present")

    # ── Step 2: Push to HF Hub ────────────────────────────
    print(f"\nStep 2: Pushing to {MODEL_REPO}...")
    best_model.push_to_hub(
        MODEL_REPO,
        token=HF_TOKEN,
        commit_message=f"Upload best model: {best_exp} (score: {best_score:.1f}%)",
    )
    tokenizer.push_to_hub(
        MODEL_REPO,
        token=HF_TOKEN,
    )

    print(f"\n✅ Model saved successfully!")
    print(f"✅ Experiment: {best_exp}")
    print(f"✅ Score:      {best_score:.1f}%")
    print(f"✅ View at:    https://huggingface.co/{MODEL_REPO}")
    print(f"\nFiles tab should now show adapter_config.json ✅")

Best experiment: exp4_lr2e4_epochs25_r32
Best score:      36.4%
Saving to:       ninadp/marathi-mitra-phi3

Step 1: Saving adapter locally to ./results/exp4_lr2e4_epochs25_r32_final...
Files saved: ['adapter_model.safetensors', 'adapter_config.json', 'added_tokens.json', 'tokenizer_config.json', 'special_tokens_map.json', 'tokenizer.model', 'tokenizer.json', 'README.md']
✅ adapter_config.json present

Step 2: Pushing to ninadp/marathi-mitra-phi3...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   2%|2         |  563kB / 25.2MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...puwh1ywt0/tokenizer.model: 100%|##########|  500kB /  500kB            


✅ Model saved successfully!
✅ Experiment: exp4_lr2e4_epochs25_r32
✅ Score:      36.4%
✅ View at:    https://huggingface.co/ninadp/marathi-mitra-phi3

Files tab should now show adapter_config.json ✅


In [ ]:
from huggingface_hub import list_repo_files

print("Files in ninadp/marathi-mitra-phi3:")
for f in list_repo_files("ninadp/marathi-mitra-phi3", token=HF_TOKEN):
    print(f"  {f}")

Files in ninadp/marathi-mitra-phi3:
  .gitattributes
  README.md
  adapter_config.json
  adapter_model.safetensors
  added_tokens.json
  special_tokens_map.json
  tokenizer.json
  tokenizer.model
  tokenizer_config.json
